## DATA CAPTURE FROM CAMERA

• CREATE CAMERA CLASS.   
• INITIALISE SETUP.    
• CREATE CAPTURE FUNCTION.    
• SAVE VIDEO CAPTURED.   

In [ ]:
from robot_utils import (
    crop_frame,
    detect_yellow,
    get_rope_side
)


Import the motors module for robot movement 

In [ ]:
import ipywidgets.widgets as widgets
import motors

robot = motors.MotorsYukon(mecanum=False)
print("Robot is ready:)")

## CAMERA CLASS 

initialise camera    

Core functions:  

• capture frames: get the .avi and .bin files this way      
• start thread       
• stop thread    


In [ ]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import math
import numpy as np
import sys
import math
import threading
from traitlets.config.configurable import SingletonConfigurable
import time

import ipywidgets.widgets as widgets
from IPython.display import display

#create two widgets for the displaying of the image
display_color = widgets.Image(format='jpeg', width='45%') #determine the width of the color image
display_depth = widgets.Image(format='jpeg', width='45%')  #determine the width of the depth image
layout=widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_depth],layout=layout) #horizontal display


# display the widget
display(sidebyside) 

timestamp = time.strftime('%Y%m%d_%H%M%S')

# Define a Camera class that inherits from SingletonConfigurable
class Camera(SingletonConfigurable):
    color_value = traitlets.Any() # monitor the color_value variable
    def __init__(self):
        super(Camera, self).__init__()

        self.zed = sl.Camera()
        # Create a InitParameters object and set configuration parameters
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA #VGA(672*376), HD720(1280*720), HD1080 (1920*1080) or ...
        init_params.depth_mode = sl.DEPTH_MODE.NONE  # depth not needed

        # Open the camera
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS: #Ensure the camera has opened succesfully
            print("Camera Open : "+repr(status)+". Exit program.")
            self.zed.close()
            exit(1)

         # Create and set RuntimeParameters after opening the camera
        self.runtime = sl.RuntimeParameters()

        #flag to control the thread
        self.thread_runnning_flag = False

        # Get the height and width
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image = sl.Mat(self.width,self.height,sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

        #setup output file
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        self.color_writer = cv2.VideoWriter(f'color_video{timestamp}.avi', fourcc, 30, (672, 376))

    def _capture_frames(self): #For data capturing only

        while(self.thread_runnning_flag==True): #continue until the thread_runnning_flag is set to be False
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                
                # Retrieve Left image
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)

                self.color_value = self.image.get_data()
                self.color_value = cv2.cvtColor(self.color_value, cv2.COLOR_BGRA2BGR)

                 # Save to file
                try:
                    self.color_writer.write(self.color_value)
                except:
                    print("Error writing file")
                    
    def start(self): #start the data capture thread
        if self.thread_runnning_flag == False: #only process if no thread is running yet
            self.thread_runnning_flag=True #flag to control the operation of the _capture_frames function
            self.thread = threading.Thread(target=self._capture_frames) #link thread with the function
            self.thread.start() #start the thread

    def stop(self): #stop the data capture thread
        if self.thread_runnning_flag == True:
            self.color_writer.release()
            self.thread_runnning_flag = False #exit the while loop in the _capture_frames
            self.thread.join() #wait the exiting of the thread       

def bgr8_to_jpeg(value):#convert numpy array to jpeg coded data for displaying 
    return bytes(cv2.imencode('.jpg',value)[1])
    
#create a camera object
camera = Camera()
camera.start() # start capturing the data

#Convert a NumPy array to JPEG-encoded data for display
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg',value)[1])


In [ ]:
# Link camera feed to display widget
camera.observe(lambda change: update_display(change['new']), names=['color_value'])

def update_display(frame):
    if frame is not None:
        display_color.value = bgr8_to_jpeg(frame)

In [ ]:
import torch
from torchvision import transforms

LABEL_NAMES = ['left', 'right', 'straight']  # alphabetical — matches ImageFolder

infer_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def classify_frame(model, frame, device, threshold=0.5):
    """
    frame: BGR numpy array from ZED camera
    returns: label string, confidence float, raw probs
    """
    if frame.shape[2]== 4:
        frame = frame[:, :, :3]
    tensor = infer_transform(frame).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    confidence, idx = probs.max(0)
    return LABEL_NAMES[idx.item()], confidence.item(), probs.cpu().numpy()

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from pathlib import Path

# --- Config ---
DATA_DIR   = Path("dataset/split")
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() 
                          else "cuda" if torch.cuda.is_available() 
                          else "cpu")
print(f"Using device: {DEVICE}")

# --- Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Datasets ---
'''
train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
'''

# --- Model ---
class LaneClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.load_state_dict(torch.load(
            #REPO_ROOT / 
            "resnet18-f37072fd.pth",
            map_location="cpu"
        ))
        
        for param in backbone.parameters():
            param.requires_grad = False
            
        in_features = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

In [ ]:
from collections import deque
import cv2

def run_resnet_only(model, robot, device,
                    base_speed           = 0.35,
                    spin_speed           = 0.12,
                    conf_threshold       = 0.85,
                    micro_threshold      = 0.10,
                    micro_frames         = 2,
                    max_spin_frames      = 35,
                    align_threshold      = 0.20,
                    consecutive_required = 9):

    model.eval()

    score_history         = deque(maxlen=micro_frames)
    consecutive_agreement = 0
    align_count           = 0
    in_corner             = False
    corner_frames         = 0
    last_turn             = None
    normal_frames         = 0
    last_rope_side        = None

    print("Starting — ResNet only")

    try:
        while True:
            t_start = time.time()

            frame = camera.color_value
            if frame is None:
                continue
            if frame.shape[2] == 4:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2BGR)

            # --- ResNet ---
            label, conf, _ = classify_frame(model, frame, device)

            # --- Pixel side only — no line fit ---
            #crops frame, detects yellow and determines rope side
            cropped   = crop_frame(frame)
            mask      = detect_yellow(cropped)
            rope_side = get_rope_side(mask)

            if rope_side is not None:
                last_rope_side = rope_side

            fallback = rope_side or last_rope_side or last_turn or 'right'

            # --- Consecutive agreement — ResNet + rope_side ---
            if rope_side is not None and \
               label in ('left', 'right') and \
               rope_side == label and \
               conf >= conf_threshold:
                consecutive_agreement += 1
                last_turn = label
            else:
                consecutive_agreement = 0

            # --- Normal frame counter ---
            if not in_corner:
                normal_frames += 1
            else:
                normal_frames = 0

            # --- Corner trigger ---
            corner_trigger = (
                consecutive_agreement >= consecutive_required and
                not in_corner
            )

            # ============================================================
            # STATE
            # ============================================================
            if not in_corner and corner_trigger:
                in_corner             = True
                corner_frames         = 0
                align_count           = 0
                consecutive_agreement = 0
                print(f"CORNER → {last_turn} | conf:{conf:.2f}")

            if in_corner:
                corner_frames += 1

                if last_turn == 'left':
                    robot.spinLeft(speed=spin_speed)
                else:
                    robot.spinRight(speed=spin_speed)

                # Exit when ResNet says straight confidently
                if conf >= conf_threshold and label == 'straight':
                    align_count += 1
                    if align_count >= 3:
                        in_corner     = False
                        align_count   = 0
                        normal_frames = 0
                        last_turn     = None
                        print("EXIT corner")
                else:
                    align_count = 0

                # Fallback if spins for too long without good alignment
                if corner_frames > max_spin_frames:
                    in_corner     = False
                    align_count   = 0
                    normal_frames = 0
                    print(f"SPIN TIMEOUT | fallback:{fallback}")

            else:
                # ResNet drives straight following
                #ResNet micro-correction to align robot to left or right
                if conf >= conf_threshold and label == 'left':
                    robot.left(speed=base_speed * 0.8)
                elif conf >= conf_threshold and label == 'right':
                    robot.right(speed=base_speed * 0.8)
                else:
                    robot.forward(speed=base_speed)

            print(f"RESNET_ONLY | {'CORNER' if in_corner else 'NORMAL'} | "
                  f"{label}({conf:.2f}) | rope:{rope_side} | "
                  f"agree:{consecutive_agreement} | last_turn:{last_turn}")

            elapsed = time.time() - t_start
            time.sleep(max(0, 0.1 - elapsed))

    except KeyboardInterrupt:
        pass
    finally:
        robot.stop()
        print("Stopped")

In [ ]:
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() 
                          else "cuda" if torch.cuda.is_available() 
                          else "cpu")
model = LaneClassifier(num_classes=3).to(DEVICE)
# Load best weights
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

display(sidebyside)
run_resnet_only(model, robot, DEVICE)